### Start vLLM server (run in a separate terminal before executing inference cells)

```bash
vllm serve /gpfs/projects/bsc100/models/DeepSeek-R1-Distill-Qwen-32B \
    --tensor-parallel-size 4 \
    --max-model-len 32000 \
    --enable-chunked-prefill \
    --max-num-batched-tokens 32000 \
    --reasoning-parser deepseek_r1 \
    --gpu-memory-utilization 0.8 \
    --port 8000
```

Wait until you see `INFO: Application startup complete.` before running the cells below.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start_path=None, marker=".git"):
    path = Path(start_path or Path.cwd()).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError("Project root not found")

ROOT_DIR = find_project_root()
print(ROOT_DIR)

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

In [ ]:
from openai import OpenAI
import httpx

MODEL_PATH = "/gpfs/projects/bsc100/models/DeepSeek-R1-Distill-Qwen-32B"
INFERENCE_PARAMS = {
    "temperature": 0.4,
    "top_p": 0.90,
    "max_tokens": 4096,
    "extra_body": {"repetition_penalty": 1.1}
}

client = OpenAI(base_url="http://localhost:8000/v1", api_key="none", http_client=httpx.Client(proxy=None))

def inference_llm(prompt):
    response = client.chat.completions.create(
        model=MODEL_PATH,
        messages=[{"role": "user", "content": prompt}],
        **INFERENCE_PARAMS
    )
    think = response.choices[0].message.reasoning_content
    answer = response.choices[0].message.content
    return think, answer

In [ ]:
import json
from pathlib import Path

# Path to the Catalonia subgraph JSON file
json_path = Path("/home/cambria/gram3/LawGraph/Spain/Catalonia/extracted_subgraph.json")

# Load the graph data
print(f"Loading graph from {json_path}...")
with open(json_path, "r", encoding="utf-8") as f:
    graph_data = json.load(f)

# Filter nodes that have textEs or textCa properties
nodes_with_text = []
for node in graph_data.get("nodes", []):
    props = node.get("properties", {})
    if "textEs" in props or "textCa" in props:
        nodes_with_text.append(node)

print(f"Total nodes loaded: {len(graph_data.get('nodes', []))}")
print(f"Nodes with text properties: {len(nodes_with_text)}")

# Show a preview of the first 5 nodes containing text properties
print("\n--- Preview of First 5 Nodes with Text ---")
for i, node in enumerate(nodes_with_text[:5]):
    props = node.get("properties", {})
    node_id = node.get("id")
    title = props.get("titleEs") or props.get("titleCa") or "No Title"
    print(f"\nIndex: {i} | Node ID: {node_id}")
    print(f"Title: {title}")
    print(f"Labels: {node.get('labels')}")
    if "textEs" in props:
        print(f"  textEs (Spanish) preview: {props['textEs'][:120]}...")
    if "textCa" in props:
        print(f"  textCa (Catalan) preview: {props['textCa'][:120]}...")

In [ ]:
# Select a node by index to infer something on
selected_index = 0

if selected_index < len(nodes_with_text):
    node = nodes_with_text[selected_index]
    props = node.get("properties", {})
    
    # Extract text and details
    text_to_analyze = props.get("textEs") or props.get("textCa")
    lang = "Spanish (textEs)" if "textEs" in props else "Catalan (textCa)"
    title = props.get("titleEs") or props.get("titleCa") or "No Title"
    node_id = node.get("id")
    labels = node.get("labels", [])
    
    # Custom inference query / question
    inference_query = "Provide a summary of the main legal actions or decrees in this text, and list any entities (e.g., people, organizations, laws) mentioned."
    
    prompt = f"""You are analyzing a node from a legal graph.
Node ID: {node_id}
Labels: {labels}
Title: {title}
Language of source text: {lang}

Source Legal Text:
\"\"\"
{text_to_analyze}
\"\"\"

Task:
{inference_query}
"""
    
    print(f"--- Prompting model on Node ID {node_id} ({title}) ---")
    print(f"Query: {inference_query}\n")
    
    # Run the inference call
    think, answer = inference_llm(prompt)
    
    print("=== Thinking Process ===")
    print(think)
    print("\n=== Model Answer ===")
    print(answer)
else:
    print(f"Error: Selected index {selected_index} is out of bounds (0 to {len(nodes_with_text)-1}).")